# 03 — Baseline & Evaluation Harness (Recall@10 / NDCG@10)

Defines the evaluation protocol every model in this project is judged by, and establishes
the **popularity baseline** any real model must beat.

**Protocol (He et al., NCF 2017):** for each user, rank the 1 held-out positive against
**99 sampled negatives** the user never interacted with. A model that puts the true item
in the top 10 of those 100 scores a hit.
- **Recall@10 = Hit@10** (single relevant item): fraction of users whose true item is top-10.
- **NDCG@10** rewards ranking the true item *higher* within the top 10.

The candidate set is built **once** and reused by every model, so comparisons are fair.

In [1]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, scipy.sparse as sp
import recsys_utils as ru
ART = '../artifacts'
train_mat = sp.load_npz(f'{ART}/train_mat.npz')
tst = np.load(f'{ART}/test.npz')
test_u, test_i = tst['u'], tst['i']
n_users, n_items = train_mat.shape
print(f'{n_users:,} users | {n_items:,} items | {len(test_u):,} test users')

162,541 users | 55,413 items | 162,495 test users


## Build the shared candidate set (1 positive + 99 negatives per user)

In [2]:
t = __import__('time').time()
users, cands = ru.build_eval_candidates(test_u, test_i, train_mat, n_items, seed=0)
np.savez(f'{ART}/cands.npz', users=users, cands=cands)
print('candidates:', cands.shape, f'(built in {__import__("time").time()-t:.1f}s)')
print('col 0 is the true item; cols 1..99 are sampled negatives')

candidates: (162495, 100) (built in 5.8s)
col 0 is the true item; cols 1..99 are sampled negatives


## Popularity baseline

Score every candidate by how many times it was interacted with in training. This is the
'recommend the most popular movies to everyone' strategy — no personalization at all.

In [3]:
item_pop = np.asarray(train_mat.getnnz(axis=0)).ravel().astype(np.float32)
pop_scores = ru.score_popularity(item_pop, cands)
recall, ndcg = ru.score_metrics(pop_scores, k=10)
print(f'Popularity baseline  ->  Recall@10 = {recall:.4f}   NDCG@10 = {ndcg:.4f}')

Popularity baseline  ->  Recall@10 = 0.9251   NDCG@10 = 0.6606


> Note: this baseline scores high because random negatives are usually obscure movies, so a
> popular held-out movie beats them easily. This is a known property of the *sampled*-negative
> protocol (Rendle et al., 2020) — the real test is how much a personalized model beats it.